### Minutas de la Junta de Gobierno

Para scrapear y crear la base de **minutas** (2011 a hoy). La misma página de Banxico
también publica las transcripciones (2018–2023); aquí se descartan, no se usan en el índice.

Las minutas se fechan por la **decisión** que discuten, no por su publicación (~2 semanas después):
`banxico_min_2023_08_10` corresponde al anuncio `banxico_apm_2023_08_10`.

Notas sobre el corpus:
- De cada minuta se conserva solo la discusión de la Junta (`keep_board_discussion`): se elimina la presentación
  del staff (sección 2 hasta 2018, ANEXO después), la portada y la lista de asistentes.
- `banxico_min_2018_11_15` tiene la fuente del PDF rota (texto ilegible, sin espacios); queda vacía y se excluye.
  Por eso hay 127 minutas en la página y 126 en el índice.
- Se extrae con `CAPTION_ONLY_MARKERS`: el filtro por defecto tiraba párrafos enteros cuando el PDF pega
  "Fuente: ..." al párrafo siguiente (~7% del texto de las minutas).

Descarga ~90 MB de PDFs a `data/raw/` (no vienen incluidos) y escribe en `data/processed/banxico/`:
`banxico_minutas_{metadata.csv, pages, documents, chunks}`.

**Fecha de corte.** `FECHA_CORTE` fija el corpus al de los resultados (decisiones hasta
septiembre de 2026). El scraper lee la página de Banxico *hoy*, así que sin el corte
entrarían anuncios nuevos que no tienen calificación del LLM y fallarían los `assert`.
Para actualizar el corpus: subir la fecha, quitar o ajustar los `assert` de conteo y
calificar los chunks nuevos con `scripts/score_chunks_openrouter.py` (cuesta dinero).

In [1]:
import sys
from pathlib import Path

import pandas as pd

# El código vive en src/, agrupado en corpus/ index/ analysis/ (sin __init__.py):
# hay que agregar cada subcarpeta a la ruta de Python. Ver src/paths.py.
sys.path.insert(0, str(Path.cwd().parent / "src"))
import paths
paths.add_src_subfolders_to_path()

from scraping import download_pdfs
from banxico_minutas import scrape_index, CAPTION_ONLY_MARKERS, keep_board_discussion
from processing import extract_pdf_pages, flag_page_quality
from chunking import create_document_chunks, split_chunks_by_tokens

In [2]:
PDF_DIR = paths.RAW_DIR / "minutas" / "pdfs"
PROCESSED_DIR = paths.PROCESSED_DIR

PDF_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FECHA_CORTE = "2026-09-30"   # ver nota arriba

In [3]:
metadata = scrape_index()
print("en la página de Banxico hoy:", metadata["document_type"].value_counts().to_dict())

# solo minutas (no transcripciones), hasta la fecha de corte, por fecha de decisión
metadata = metadata[(metadata["document_type"] == "Minuta")
                    & (metadata["decision_date"] <= pd.Timestamp(FECHA_CORTE))].reset_index(drop=True)
print("minutas hasta la fecha de corte:", metadata.shape)
metadata.head()

en la página de Banxico hoy: {'Minuta': 127, 'Transcripción': 45}
minutas hasta la fecha de corte: (127, 13)


,document_id,institution,document_type,document_variant,title,publication_date,publication_month,decision_date,reference_year,quarter,resource_type,pdf_url,source_page_url
0,banxico_min_2026_09_24,Banco de México,Minuta,minutes,Minuta de la reunión de la Junta de Gobierno d...,2026-10-08,2026-10,2026-09-24,2026,3,Minuta,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
1,banxico_min_2026_08_06,Banco de México,Minuta,minutes,Minuta de la reunión de la Junta de Gobierno d...,2026-08-20,2026-08,2026-08-06,2026,3,Minuta,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
2,banxico_min_2026_06_25,Banco de México,Minuta,minutes,Minuta de la reunión de la Junta de Gobierno d...,2026-07-09,2026-07,2026-06-25,2026,2,Minuta,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
3,banxico_min_2026_05_07,Banco de México,Minuta,minutes,Minuta de la reunión de la Junta de Gobierno d...,2026-05-21,2026-05,2026-05-07,2026,2,Minuta,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
4,banxico_min_2026_03_26,Banco de México,Minuta,minutes,Minuta de la reunión de la Junta de Gobierno d...,2026-04-09,2026-04,2026-03-26,2026,1,Minuta,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...


In [4]:
metadata["reference_year"].value_counts().sort_index()

reference_year
2011    8
2012    8
2013    8
2014    8
2015    8
2016    8
2017    8
2018    8
2019    8
2020    9
2021    8
2022    8
2023    8
2024    8
2025    8
2026    6
Name: count, dtype: int64

In [5]:
download_results = download_pdfs(metadata, PDF_DIR)
download_results["download_status"].value_counts()

  0%|          | 0/127 [00:00<?, ?it/s]

download_status
already_exists    127
Name: count, dtype: int64

In [6]:
metadata = metadata.merge(download_results, on="document_id", how="left")

assert metadata["document_id"].is_unique
assert metadata["pdf_url"].is_unique
assert len(metadata) == 127

metadata.to_csv(PROCESSED_DIR / "banxico_minutas_metadata.csv", index=False)

In [7]:
# extracción de texto con el filtro de pies de gráfica más estricto (ver nota arriba)
pages = extract_pdf_pages(metadata, column_options={"noise_markers": CAPTION_ONLY_MARKERS})
print(pages.shape)
assert pages["document_id"].nunique() == 127

  0%|          | 0/127 [00:00<?, ?it/s]

(3079, 13)


In [8]:
# las columnas de metadatos que se copian a documentos y chunks
document_metadata_columns = [
    "document_id",
    "institution",
    "document_type",
    "document_variant",
    "title",
    "publication_date",
    "publication_month",
    "decision_date",
    "reference_year",
    "quarter",
    "pdf_url",
]
chunk_metadata_columns = document_metadata_columns[1:]


def build_documents(usable_pages, document_metadata):
    """Una fila por documento: el texto de sus páginas utilizables unido, más metadatos."""
    document_texts = (
        usable_pages
        .groupby("document_id", as_index=False, sort=False)
        .agg(
            document_text=("page_text", lambda values: "\n\n".join(values)),
            n_pages=("page_number", "nunique"),
            total_characters=("n_characters", "sum"),
        )
    )
    documents = document_metadata[document_metadata_columns].merge(
        document_texts, on="document_id", how="inner", validate="one_to_one")
    documents["n_words"] = documents["document_text"].str.split().str.len()
    return documents

In [9]:
pages = keep_board_discussion(pages)
pages = flag_page_quality(pages)
pages[["is_near_empty", "has_encoding_error", "is_usable_page"]].sum()

is_near_empty         1981
has_encoding_error       0
is_usable_page        1155
dtype: int64

In [10]:
pages.to_parquet(PROCESSED_DIR / "banxico_minutas_pages.parquet", index=False)

usable = pages.loc[pages["is_usable_page"]].sort_values(["document_id", "page_number"])

# la única minuta sin discusión encontrada debe ser la del PDF ilegible
missing = set(metadata["document_id"]) - set(usable["document_id"])
print("Minutas sin discusión de la Junta:", missing)
assert missing == {"banxico_min_2018_11_15"}

Minutas sin discusión de la Junta: {'banxico_min_2018_11_15'}


In [11]:
documents = build_documents(usable, metadata)
documents.to_parquet(PROCESSED_DIR / "banxico_minutas_documents.parquet", index=False)

# las minutas se volvieron mucho más largas con el tiempo: palabras (mediana) por año
documents.groupby("reference_year")["n_words"].median()

reference_year
2011    3852.0
2012    3894.0
2013    4781.0
2014    4822.5
2015    5444.0
2016    5506.5
2017    7066.0
2018    7242.0
2019    9109.5
2020    7421.0
2021    7953.5
2022    7047.0
2023    7389.5
2024    8200.0
2025    8205.5
2026    9245.5
Name: n_words, dtype: float64

#### Chunks

Primero pedazos de hasta ~3,000 caracteres cortados en párrafos (los que lee el LLM, *chunks padre*).
Luego cada uno se parte en sub-chunks de ≤ 450 tokens, porque el modelo de embeddings (método 2) no acepta
textos más largos. `rebuild_parent_chunks` (en `src/index/common.py`) los vuelve a juntar para el LLM.

In [12]:
## AGREGADO PARA PODER FIT EL MODELO DE EMBEDDING
from transformers import AutoTokenizer

MODEL_NAME = "intfloat/multilingual-e5-base"
MAX_TOKENS = 450

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [13]:
chunks = create_document_chunks(
    pages_df=usable,
    metadata_df=metadata,
    metadata_columns=chunk_metadata_columns)
n_parents = len(chunks)

chunks = split_chunks_by_tokens(chunks, tokenizer, max_tokens=MAX_TOKENS)
n_tokens = chunks["chunk_text"].apply(
    lambda text: len(tokenizer(f"passage: {text}", add_special_tokens=True,
                               truncation=False, verbose=False)["input_ids"]))
assert n_tokens.max() <= 512

chunks.to_parquet(PROCESSED_DIR / "banxico_minutas_chunks.parquet", index=False)
print(f"{n_parents} chunks padre, {len(chunks)} sub-chunks, "
      f"{n_parents / chunks['document_id'].nunique():.1f} chunks padre por minuta")

  0%|          | 0/126 [00:00<?, ?it/s]

  0%|          | 0/2113 [00:00<?, ?it/s]

2113 chunks padre, 4003 sub-chunks, 16.8 chunks padre por minuta
